In [1]:
import datasets as ds
import torch
import tqdm
import peft

from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login

import help_lib as hb


login()

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'cuda'

In [2]:
STUDENT_MODEL = "google/gemma-3-1b-it"
test_dataset = ds.load_from_disk("data/test_dataset")

In [3]:
def get_pretrained_lora_model(pretrain_model_path, is_trainable=True):
    student_model = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL)
    model = peft.PeftModel.from_pretrained(
        student_model,
        pretrain_model_path,
        is_trainable=is_trainable
    )
    
    if not is_trainable:
        model.eval()
    model.to(device)
    
    return model

def generate_with_k_attempts_in_batches(
    model,
    tokenizer,
    input_dataset,
    batch_size,
    k,
    max_new_tokens,
    temperature,
    top_p,
    device=device
):
    model.eval()
    loader = DataLoader(input_dataset, batch_size=batch_size, collate_fn=hb.collate_data_fn)

    all_answers = []
    for batch in tqdm.tqdm(loader):
        batch_messages = batch["messages"]
        batch_nums = batch["nums"]
        batch_targets = batch["target"]

        batch_prompts = [
            tokenizer.apply_chat_template(
                m,
                tokenize=False,
                add_generation_prompt=True,
            )
            for m in batch_messages
        ]

        enc = tokenizer(
            batch_prompts,
            padding=True,
            truncation=True,
            return_tensors="pt",
            add_special_tokens=False,
        )
        enc = {key: value.to(device) for key, value in enc.items()}
        
        input_len = enc["input_ids"].shape[1]
        rep_enc = {
            key: value.repeat_interleave(k, dim=0)
            for key, value in enc.items()
        }

        with torch.inference_mode():
            out = model.generate(
                **rep_enc,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                max_new_tokens=max_new_tokens,
            )

        # [prompt1_var1, prompt1_var2, ..., prompt1_vark, prompt2_var1, ...]
        for j in range(len(batch_messages)):
            nums = batch_nums[j]
            target = batch_targets[j]

            chosen_answer = ""

            start = j * k
            end = start + k

            for seq in out[start:end]:
                gen_text = tokenizer.decode(
                    seq[input_len:],
                    skip_special_tokens=True
                ).strip()

                
                if "<answer>" in gen_text:
                    gen_text = hb.extract_last_answer(gen_text)
                
                candidate = hb.normalize_text(hb.extract_generated_equation(gen_text))
                if not candidate:
                    continue
                
                chosen_answer = candidate
                if hb.is_valid_equation(candidate, nums, target):
                    break

            all_answers.append(chosen_answer)

    return all_answers

In [4]:
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
model = get_pretrained_lora_model(pretrain_model_path="./contest/gemma3_62", is_trainable=False)

Loading weights: 100%|██████████| 340/340 [00:00<00:00, 27258.13it/s]


In [5]:
all_responses = generate_with_k_attempts_in_batches(
    model=model,
    tokenizer=tokenizer,
    input_dataset=test_dataset,
    batch_size=128,
    k=16,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.95
)

100%|██████████| 16/16 [16:53<00:00, 63.33s/it]


Посчитаем Accuracy

In [12]:
acc = (sum(1 for ans, el in zip(all_responses, test_dataset) if hb.is_valid_equation(ans, el["nums"], el["target"]) ) 
 / len(all_responses))

print(f"test accuracy: {acc}")

test accuracy: 0.6005


In [7]:
answers = [r.split('\n')[-1].split("=")[0].strip() for r in all_responses]
data = [
    {
        "id": i,
        "equation": answer if answer else "1 + 1"
    }
    for i, answer in enumerate(answers)
]

In [8]:
ds.Dataset.from_list(data).to_csv("contest/submissions/submission.csv")

Creating CSV from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 91.14ba/s]


44090